In [2]:
# ============================================================
# Temporal Data Preparation - Imports
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:
# ============================================================
# Load Required Datasets
# ============================================================

VALUATIONS_PATH = '../../../data/raw/player_valuations.csv'
APPEARANCES_PATH = '../../../data/raw/appearances.csv'
PLAYERS_PATH = '../../../data/raw/players.csv'

df_valuations = pd.read_csv(VALUATIONS_PATH)
df_appearances = pd.read_csv(APPEARANCES_PATH)
df_players = pd.read_csv(PLAYERS_PATH)

print("Datasets loaded successfully.\n")

print(
    f"Player valuations : {df_valuations.shape}"
)

print(
    f"Appearances       : {df_appearances.shape}"
)

print(
    f"Players           : {df_players.shape}"
)

Datasets loaded successfully.

Player valuations : (640176, 6)
Appearances       : (1884051, 13)
Players           : (47669, 26)


In [4]:
# ============================================================
# Temporal Configuration
# ============================================================

df_valuations['date'] = pd.to_datetime(
    df_valuations['date'],
    errors='coerce'
)

df_appearances['date'] = pd.to_datetime(
    df_appearances['date'],
    errors='coerce'
)

SELECTED_HORIZON_MONTHS = 6
SELECTED_TOLERANCE_DAYS = 45

print("=" * 60)
print("TEMPORAL CONFIGURATION")
print("=" * 60)

print(
    f"Forecast horizon : "
    f"{SELECTED_HORIZON_MONTHS} months"
)

print(
    f"Target tolerance : "
    f"±{SELECTED_TOLERANCE_DAYS} days"
)

print("\nValuation range:")
print(
    df_valuations['date'].min(),
    "→",
    df_valuations['date'].max()
)

print("\nAppearance range:")
print(
    df_appearances['date'].min(),
    "→",
    df_appearances['date'].max()
)

TEMPORAL CONFIGURATION
Forecast horizon : 6 months
Target tolerance : ±45 days

Valuation range:
2000-01-20 00:00:00 → 2026-05-18 00:00:00

Appearance range:
2012-07-03 00:00:00 → 2026-05-18 00:00:00


In [5]:
# ============================================================
# Create Candidate Temporal Snapshots
# ============================================================

first_appearance = (
    df_appearances
    .groupby('player_id')['date']
    .min()
    .reset_index()
    .rename(
        columns={
            'date': 'first_appearance_date'
        }
    )
)

snapshots = (
    df_valuations
    .merge(
        first_appearance,
        on='player_id',
        how='inner'
    )
)

# Keep only valuations occurring on or after
# the player's first recorded appearance
snapshots = snapshots[
    snapshots['date']
    >= snapshots['first_appearance_date']
].copy()

snapshots = snapshots.rename(
    columns={
        'date': 'snapshot_date',
        'market_value_in_eur':
            'current_market_value'
    }
)

print("=" * 60)
print("CANDIDATE TEMPORAL SNAPSHOTS")
print("=" * 60)

print(
    f"Snapshots : {len(snapshots):,}"
)

print(
    f"Players   : "
    f"{snapshots['player_id'].nunique():,}"
)

print("\nSnapshot date range:")

print(
    snapshots['snapshot_date'].min(),
    "→",
    snapshots['snapshot_date'].max()
)

display(
    snapshots[
        [
            'player_id',
            'snapshot_date',
            'current_market_value',
            'current_club_name'
        ]
    ].head()
)

CANDIDATE TEMPORAL SNAPSHOTS
Snapshots : 329,104
Players   : 27,690

Snapshot date range:
2012-07-06 00:00:00 → 2026-05-18 00:00:00


,player_id,snapshot_date,current_market_value,current_club_name
74529,4520,2012-07-06,2500000,Twente Enschede FC
74538,30003,2012-07-06,5000000,Twente Enschede FC
74546,47740,2012-07-06,400000,Twente Enschede FC
74552,60312,2012-07-06,600000,Twente Enschede FC
74565,80073,2012-07-06,1500000,Twente Enschede FC


In [ ]:
# ============================================================
# Create Six-Month Future Market Value Target
# ============================================================

# Keep the columns needed from the snapshot table
snapshot_targets = snapshots[
    [
        'player_id',
        'snapshot_date',
        'current_market_value'
    ]
].copy()

# Ideal prediction date = snapshot + 6 months
snapshot_targets['target_date'] = (
    snapshot_targets['snapshot_date']
    + pd.DateOffset(months=SELECTED_HORIZON_MONTHS)
)

# Acceptable target window
snapshot_targets['window_start'] = (
    snapshot_targets['target_date']
    - pd.Timedelta(days=SELECTED_TOLERANCE_DAYS)
)

snapshot_targets['window_end'] = (
    snapshot_targets['target_date']
    + pd.Timedelta(days=SELECTED_TOLERANCE_DAYS)
)

# Prepare all historical valuation records
future_valuations = (
    df_valuations[
        [
            'player_id',
            'date',
            'market_value_in_eur'
        ]
    ]
    .rename(
        columns={
            'date': 'future_date',
            'market_value_in_eur':
                'future_market_value'
        }
    )
    .copy()
)

# Generate candidate future valuations for the same player
target_candidates = snapshot_targets.merge(
    future_valuations,
    on='player_id',
    how='left'
)

# Keep only valuations:
# 1. after the snapshot
# 2. inside the 6-month ±45-day target window
target_candidates = target_candidates[
    (target_candidates['future_date']
        > target_candidates['snapshot_date'])
    &
    (target_candidates['future_date']
        >= target_candidates['window_start'])
    &
    (target_candidates['future_date']
        <= target_candidates['window_end'])
].copy()

# Calculate distance from the ideal 6-month target date
target_candidates['target_distance_days'] = (
    target_candidates['future_date']
    - target_candidates['target_date']
).abs().dt.days

# If several valuations exist inside the window,
# select the one closest to the ideal target date
target_candidates = (
    target_candidates
    .sort_values(
        [
            'player_id',
            'snapshot_date',
            'target_distance_days'
        ]
    )
    .drop_duplicates(
        subset=[
            'player_id',
            'snapshot_date'
        ],
        keep='first'
    )
)

print("=" * 70)
print("FUTURE TARGET CREATION")
print("=" * 70)

print(
    f"Candidate snapshots : "
    f"{len(snapshot_targets):,}"
)

print(
    f"Matched targets     : "
    f"{len(target_candidates):,}"
)

print(
    f"Players             : "
    f"{target_candidates['player_id'].nunique():,}"
)

print(
    f"Coverage            : "
    f"{len(target_candidates) / len(snapshot_targets) * 100:.2f}%"
)

print(
    f"Median target distance: "
    f"{target_candidates['target_distance_days'].median():.1f} days"
)

FUTURE TARGET CREATION
Candidate snapshots : 329,104
Matched targets     : 213,905
Players             : 24,275
Coverage            : 65.00%
Median target distance: 18.0 days


In [7]:
# ============================================================
# Validate Future Market Value Targets
# ============================================================

future_after_snapshot = (
    target_candidates['future_date']
    > target_candidates['snapshot_date']
).all()

inside_tolerance = (
    target_candidates['target_distance_days']
    <= SELECTED_TOLERANCE_DAYS
).all()

missing_targets = (
    target_candidates['future_market_value']
    .isna()
    .sum()
)

duplicate_snapshots = (
    target_candidates
    .duplicated(
        subset=[
            'player_id',
            'snapshot_date'
        ]
    )
    .sum()
)

print("=" * 70)
print("FUTURE TARGET VALIDATION")
print("=" * 70)

print(
    f"All future dates after snapshot : "
    f"{future_after_snapshot}"
)

print(
    f"All targets within tolerance    : "
    f"{inside_tolerance}"
)

print(
    f"Missing future market values    : "
    f"{missing_targets:,}"
)

print(
    f"Duplicate player-date snapshots : "
    f"{duplicate_snapshots:,}"
)

print("\nTarget distance statistics:")

display(
    target_candidates[
        'target_distance_days'
    ].describe()
)

FUTURE TARGET VALIDATION
All future dates after snapshot : True
All targets within tolerance    : True
Missing future market values    : 0
Duplicate player-date snapshots : 0

Target distance statistics:


count   213,905.00
mean         19.19
std          12.67
min           0.00
25%           8.00
50%          18.00
75%          29.00
max          45.00
Name: target_distance_days, dtype: float64

In [8]:
# ============================================================
# Inspect Temporal Target Examples
# ============================================================

target_examples = target_candidates[
    [
        'player_id',
        'snapshot_date',
        'current_market_value',
        'target_date',
        'future_date',
        'future_market_value',
        'target_distance_days'
    ]
].copy()

display(
    target_examples.head(10)
)

,player_id,snapshot_date,current_market_value,target_date,future_date,future_market_value,target_distance_days
95610,10,2013-01-14,4000000,2013-07-14,2013-06-19,2000000,25
225375,10,2013-06-19,2000000,2013-12-19,2014-01-07,1000000,19
360212,10,2014-01-07,1000000,2014-07-07,2014-07-07,1000000,0
558100,10,2014-07-07,1000000,2015-01-07,2015-01-07,1000000,0
736003,10,2015-01-07,1000000,2015-07-07,2015-07-01,1000000,6
1006609,10,2015-07-01,1000000,2016-01-01,2016-01-04,1000000,3
79350,26,2013-01-13,4000000,2013-07-13,2013-06-30,5000000,13
558840,26,2014-07-10,5000000,2015-01-10,2015-02-04,3000000,25
818169,26,2015-02-04,3000000,2015-08-04,2015-07-01,2000000,34
1006632,26,2015-07-01,2000000,2016-01-01,2016-02-15,1000000,45


In [9]:
# ============================================================
# Inspect Player Attributes for Temporal Safety
# ============================================================

print("=" * 70)
print("PLAYER ATTRIBUTE AUDIT")
print("=" * 70)

print("\nAvailable player columns:")
print(df_players.columns.tolist())

player_audit_columns = [
    col for col in [
        'player_id',
        'name',
        'date_of_birth',
        'position',
        'sub_position',
        'foot',
        'height_in_cm',
        'country_of_citizenship',
        'current_club_id',
        'current_club_name',
        'highest_market_value_in_eur'
    ]
    if col in df_players.columns
]

print("\nSelected columns for inspection:")

display(
    df_players[
        player_audit_columns
    ].head(10)
)

PLAYER ATTRIBUTE AUDIT

Available player columns:
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

Selected columns for inspection:


,player_id,name,date_of_birth,position,sub_position,foot,height_in_cm,country_of_citizenship,current_club_id,current_club_name,highest_market_value_in_eur
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,Centre-Forward,right,184.00,Germany,398,Società Sportiva Lazio S.p.A.,"30,000,000.00"
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,Goalkeeper,left,190.00,Germany,16,Borussia Dortmund,"8,000,000.00"
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,Centre-Forward,NaN,NaN,Bulgaria,1091,Panthessalonikios Athlitikos Omilos Konstantin...,"34,500,000.00"
3,77,Lúcio,1978-05-08 00:00:00,Defender,Centre-Back,NaN,NaN,Brazil,506,Juventus Football Club,"24,500,000.00"
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,Goalkeeper,right,194.00,Germany,27,FC Bayern München,"3,000,000.00"
5,109,Dedê,1978-04-18 00:00:00,Defender,Left-Back,NaN,NaN,Brazil,825,Eskisehirspor,"9,500,000.00"
6,123,Christoph Metzelder,1980-11-05 00:00:00,Defender,Centre-Back,NaN,NaN,Germany,33,FC Schalke 04,"9,500,000.00"
7,132,Tomas Rosicky,1980-10-04 00:00:00,Midfield,Attacking Midfield,both,179.00,Czech Republic,11,Arsenal Football Club,"17,500,000.00"
8,162,Marc Ziegler,1976-06-13 00:00:00,Goalkeeper,Goalkeeper,right,193.00,Germany,79,Verein für Bewegungsspiele Stuttgart 1893,"1,250,000.00"
9,215,Roque Santa Cruz,1981-08-16 00:00:00,Attack,Centre-Forward,right,193.00,Paraguay,1084,Málaga CF,"12,000,000.00"


In [10]:
# ============================================================
# Create Age at Snapshot
# ============================================================

df_players['date_of_birth'] = pd.to_datetime(
    df_players['date_of_birth'],
    errors='coerce'
)

player_static = df_players[
    [
        'player_id',
        'date_of_birth'
    ]
].copy()

temporal_dataset = target_candidates.merge(
    player_static,
    on='player_id',
    how='left'
)

temporal_dataset['age_at_snapshot'] = (
    (
        temporal_dataset['snapshot_date']
        - temporal_dataset['date_of_birth']
    ).dt.days
    / 365.25
)

print("=" * 70)
print("AGE AT SNAPSHOT")
print("=" * 70)

print(
    f"Rows                : "
    f"{len(temporal_dataset):,}"
)

print(
    f"Missing DOB         : "
    f"{temporal_dataset['date_of_birth'].isna().sum():,}"
)

print(
    f"Missing age         : "
    f"{temporal_dataset['age_at_snapshot'].isna().sum():,}"
)

print("\nAge statistics:")

display(
    temporal_dataset[
        'age_at_snapshot'
    ].describe()
)

AGE AT SNAPSHOT
Rows                : 213,905
Missing DOB         : 203
Missing age         : 203

Age statistics:


count   213,702.00
mean         26.72
std           4.41
min          15.38
25%          23.32
50%          26.46
75%          29.89
max          44.90
Name: age_at_snapshot, dtype: float64

In [11]:
# ============================================================
# Validate Age at Snapshot
# ============================================================

print("=" * 70)
print("AGE VALIDATION")
print("=" * 70)

print(
    f"Minimum age : "
    f"{temporal_dataset['age_at_snapshot'].min():.2f}"
)

print(
    f"Median age  : "
    f"{temporal_dataset['age_at_snapshot'].median():.2f}"
)

print(
    f"Maximum age : "
    f"{temporal_dataset['age_at_snapshot'].max():.2f}"
)

invalid_age = temporal_dataset[
    (temporal_dataset['age_at_snapshot'] < 15)
    |
    (temporal_dataset['age_at_snapshot'] > 45)
]

print(
    f"\nPotentially unusual ages (<15 or >45): "
    f"{len(invalid_age):,}"
)

display(
    temporal_dataset[
        [
            'player_id',
            'date_of_birth',
            'snapshot_date',
            'age_at_snapshot',
            'current_market_value',
            'future_market_value'
        ]
    ].head(10)
)

AGE VALIDATION
Minimum age : 15.38
Median age  : 26.46
Maximum age : 44.90

Potentially unusual ages (<15 or >45): 0


,player_id,date_of_birth,snapshot_date,age_at_snapshot,current_market_value,future_market_value
0,10,1978-06-09,2013-01-14,34.60,4000000,2000000
1,10,1978-06-09,2013-06-19,35.03,2000000,1000000
2,10,1978-06-09,2014-01-07,35.58,1000000,1000000
3,10,1978-06-09,2014-07-07,36.08,1000000,1000000
4,10,1978-06-09,2015-01-07,36.58,1000000,1000000
5,10,1978-06-09,2015-07-01,37.06,1000000,1000000
6,26,1980-08-06,2013-01-13,32.44,4000000,5000000
7,26,1980-08-06,2014-07-10,33.92,5000000,3000000
8,26,1980-08-06,2015-02-04,34.50,3000000,2000000
9,26,1980-08-06,2015-07-01,34.90,2000000,1000000


In [12]:
# ============================================================
# Inspect Appearance Performance Features
# ============================================================

print("=" * 70)
print("APPEARANCE PERFORMANCE FEATURE AUDIT")
print("=" * 70)

print("\nAppearance columns:")
print(df_appearances.columns.tolist())

performance_columns = [
    col for col in [
        'player_id',
        'date',
        'game_id',
        'goals',
        'assists',
        'minutes_played',
        'yellow_cards',
        'red_cards'
    ]
    if col in df_appearances.columns
]

print("\nPerformance columns selected for inspection:")

display(
    df_appearances[
        performance_columns
    ].head(10)
)

print("\nMissing values:")

display(
    df_appearances[
        performance_columns
    ].isna().sum()
)

APPEARANCE PERFORMANCE FEATURE AUDIT

Appearance columns:
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']

Performance columns selected for inspection:


,player_id,date,game_id,goals,assists,minutes_played,yellow_cards,red_cards
0,38004,2012-07-03,2231978,2,0,90,0,0
1,79232,2012-07-05,2233748,0,0,90,0,0
2,42792,2012-07-05,2234413,0,0,45,0,0
3,73333,2012-07-05,2234418,0,0,90,0,0
4,122011,2012-07-05,2234421,0,1,90,0,0
5,146889,2012-07-05,2234421,0,0,90,1,0
6,28716,2012-07-05,2235539,0,0,90,0,0
7,69445,2012-07-05,2235539,0,1,90,1,0
8,19409,2012-07-05,2235545,0,0,45,0,0
9,30003,2012-07-05,2235545,0,0,90,0,0



Missing values:


player_id         0
date              0
game_id           0
goals             0
assists           0
minutes_played    0
yellow_cards      0
red_cards         0
dtype: int64

In [13]:
# ============================================================
# Define Historical Performance Window
# ============================================================

PERFORMANCE_WINDOW_MONTHS = 6

temporal_dataset['performance_window_start'] = (
    temporal_dataset['snapshot_date']
    - pd.DateOffset(months=PERFORMANCE_WINDOW_MONTHS)
)

print("=" * 70)
print("HISTORICAL PERFORMANCE WINDOW")
print("=" * 70)

print(
    f"Performance lookback : "
    f"{PERFORMANCE_WINDOW_MONTHS} months"
)

display(
    temporal_dataset[
        [
            'player_id',
            'performance_window_start',
            'snapshot_date',
            'current_market_value',
            'future_market_value'
        ]
    ].head(10)
)

HISTORICAL PERFORMANCE WINDOW
Performance lookback : 6 months


,player_id,performance_window_start,snapshot_date,current_market_value,future_market_value
0,10,2012-07-14,2013-01-14,4000000,2000000
1,10,2012-12-19,2013-06-19,2000000,1000000
2,10,2013-07-07,2014-01-07,1000000,1000000
3,10,2014-01-07,2014-07-07,1000000,1000000
4,10,2014-07-07,2015-01-07,1000000,1000000
5,10,2015-01-01,2015-07-01,1000000,1000000
6,26,2012-07-13,2013-01-13,4000000,5000000
7,26,2014-01-10,2014-07-10,5000000,3000000
8,26,2014-08-04,2015-02-04,3000000,2000000
9,26,2015-01-01,2015-07-01,2000000,1000000


In [14]:
# ============================================================
# Prepare Appearance Data for Temporal Aggregation
# ============================================================

appearance_performance = df_appearances[
    [
        'player_id',
        'game_id',
        'date',
        'goals',
        'assists',
        'minutes_played',
        'yellow_cards',
        'red_cards'
    ]
].copy()

appearance_performance = (
    appearance_performance
    .sort_values(
        [
            'player_id',
            'date'
        ]
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("APPEARANCE DATA PREPARED")
print("=" * 70)

print(
    f"Appearance records : "
    f"{len(appearance_performance):,}"
)

print(
    f"Players            : "
    f"{appearance_performance['player_id'].nunique():,}"
)

print("\nDate range:")

print(
    appearance_performance['date'].min(),
    "→",
    appearance_performance['date'].max()
)

display(
    appearance_performance.head()
)

APPEARANCE DATA PREPARED
Appearance records : 1,884,051
Players            : 28,713

Date range:
2012-07-03 00:00:00 → 2026-05-18 00:00:00


,player_id,game_id,date,goals,assists,minutes_played,yellow_cards,red_cards
0,10,2254965,2012-08-23,1,0,83,1,0
1,10,2251263,2012-08-26,0,0,90,1,0
2,10,2251277,2012-09-02,2,0,89,1,0
3,10,2251283,2012-09-16,1,0,90,0,0
4,10,2262399,2012-09-20,0,0,90,0,0


In [15]:
# ============================================================
# Build Cumulative Player Performance
# ============================================================

performance_metrics = [
    'goals',
    'assists',
    'minutes_played',
    'yellow_cards',
    'red_cards'
]

# Treat missing match statistics as zero
appearance_performance[
    performance_metrics
] = (
    appearance_performance[
        performance_metrics
    ]
    .fillna(0)
)

# One appearance row represents one player appearance
appearance_performance['games'] = 1

cumulative_metrics = (
    performance_metrics
    + ['games']
)

for metric in cumulative_metrics:

    appearance_performance[
        f'cum_{metric}'
    ] = (
        appearance_performance
        .groupby('player_id')[metric]
        .cumsum()
    )

print("=" * 70)
print("CUMULATIVE PERFORMANCE CREATED")
print("=" * 70)

display(
    appearance_performance[
        [
            'player_id',
            'date',
            'games',
            'goals',
            'assists',
            'minutes_played',
            'cum_games',
            'cum_goals',
            'cum_assists',
            'cum_minutes_played'
        ]
    ].head(15)
)

CUMULATIVE PERFORMANCE CREATED


,player_id,date,games,goals,assists,minutes_played,cum_games,cum_goals,cum_assists,cum_minutes_played
0,10,2012-08-23,1,1,0,83,1,1,0,83
1,10,2012-08-26,1,0,0,90,2,1,0,173
2,10,2012-09-02,1,2,0,89,3,3,0,262
3,10,2012-09-16,1,1,0,90,4,4,0,352
4,10,2012-09-20,1,0,0,90,5,4,0,442
5,10,2012-09-23,1,0,0,14,6,4,0,456
6,10,2012-09-26,1,0,0,90,7,4,0,546
7,10,2012-09-30,1,0,1,90,8,4,1,636
8,10,2012-10-07,1,2,0,72,9,6,1,708
9,10,2012-10-20,1,1,0,90,10,7,1,798


In [16]:
# ============================================================
# Get Cumulative Performance at Snapshot Date
# ============================================================

cum_columns = [
    'cum_games',
    'cum_goals',
    'cum_assists',
    'cum_minutes_played',
    'cum_yellow_cards',
    'cum_red_cards'
]

performance_cumulative = (
    appearance_performance[
        ['player_id', 'date'] + cum_columns
    ]
    .copy()
)

# Required by merge_asof
performance_cumulative = performance_cumulative.sort_values(
    ['date', 'player_id']
).reset_index(drop=True)

snapshot_lookup = (
    temporal_dataset[
        [
            'player_id',
            'snapshot_date'
        ]
    ]
    .copy()
    .sort_values(
        ['snapshot_date', 'player_id']
    )
    .reset_index(drop=True)
)

snapshot_cumulative = pd.merge_asof(
    snapshot_lookup,
    performance_cumulative,
    left_on='snapshot_date',
    right_on='date',
    by='player_id',
    direction='backward',
    allow_exact_matches=True
)

snapshot_cumulative = snapshot_cumulative.rename(
    columns={
        'date': 'last_appearance_at_snapshot'
    }
)

print("=" * 70)
print("CUMULATIVE PERFORMANCE AT SNAPSHOT")
print("=" * 70)

print(
    f"Snapshots processed : "
    f"{len(snapshot_cumulative):,}"
)

print(
    f"Snapshots with prior appearance : "
    f"{snapshot_cumulative['last_appearance_at_snapshot'].notna().sum():,}"
)

display(
    snapshot_cumulative.head(10)
)

CUMULATIVE PERFORMANCE AT SNAPSHOT
Snapshots processed : 213,905
Snapshots with prior appearance : 213,905


,player_id,snapshot_date,last_appearance_at_snapshot,cum_games,cum_goals,cum_assists,cum_minutes_played,cum_yellow_cards,cum_red_cards
0,4520,2012-07-06,2012-07-05,1,0,0,90,0,0
1,30003,2012-07-06,2012-07-05,1,0,0,90,0,0
2,47740,2012-07-06,2012-07-05,1,0,0,90,0,0
3,60312,2012-07-06,2012-07-05,1,1,1,45,0,0
4,80073,2012-07-06,2012-07-05,1,2,1,90,0,0
5,232499,2012-07-13,2012-07-12,1,0,0,19,0,0
6,232500,2012-07-13,2012-07-12,1,0,0,28,0,0
7,14963,2012-07-18,2012-07-15,1,0,0,2,0,0
8,15765,2012-07-18,2012-07-15,1,1,0,90,1,0
9,22499,2012-07-18,2012-07-15,1,0,0,90,0,0


In [17]:
# ============================================================
# Get Cumulative Performance at Window Start
# ============================================================

window_lookup = (
    temporal_dataset[
        [
            'player_id',
            'performance_window_start'
        ]
    ]
    .copy()
    .sort_values(
        ['performance_window_start', 'player_id']
    )
    .reset_index(drop=True)
)

window_cumulative = pd.merge_asof(
    window_lookup,
    performance_cumulative,
    left_on='performance_window_start',
    right_on='date',
    by='player_id',
    direction='backward',
    allow_exact_matches=True
)

window_cumulative = window_cumulative.rename(
    columns={
        'date': 'last_appearance_at_window_start'
    }
)

rename_window_columns = {
    col: f'{col}_window_start'
    for col in cum_columns
}

window_cumulative = window_cumulative.rename(
    columns=rename_window_columns
)

print("=" * 70)
print("CUMULATIVE PERFORMANCE AT WINDOW START")
print("=" * 70)

print(
    f"Windows processed : "
    f"{len(window_cumulative):,}"
)

print(
    f"Windows with earlier appearance : "
    f"{window_cumulative['last_appearance_at_window_start'].notna().sum():,}"
)

display(
    window_cumulative.head(10)
)

CUMULATIVE PERFORMANCE AT WINDOW START
Windows processed : 213,905
Windows with earlier appearance : 194,300


,player_id,performance_window_start,last_appearance_at_window_start,cum_games_window_start,cum_goals_window_start,cum_assists_window_start,cum_minutes_played_window_start,cum_yellow_cards_window_start,cum_red_cards_window_start
0,4520,2012-01-06,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,30003,2012-01-06,NaT,NaN,NaN,NaN,NaN,NaN,NaN
2,47740,2012-01-06,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,60312,2012-01-06,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,80073,2012-01-06,NaT,NaN,NaN,NaN,NaN,NaN,NaN
5,232499,2012-01-13,NaT,NaN,NaN,NaN,NaN,NaN,NaN
6,232500,2012-01-13,NaT,NaN,NaN,NaN,NaN,NaN,NaN
7,14963,2012-01-18,NaT,NaN,NaN,NaN,NaN,NaN,NaN
8,15765,2012-01-18,NaT,NaN,NaN,NaN,NaN,NaN,NaN
9,22499,2012-01-18,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# ============================================================
# Calculate Last Six-Month Performance Features
# ============================================================

# Restore original temporal dataset order
temporal_dataset = temporal_dataset.reset_index(
    drop=True
)

snapshot_cumulative = (
    snapshot_cumulative
    .sort_values(
        ['player_id', 'snapshot_date']
    )
    .reset_index(drop=True)
)

window_cumulative = (
    window_cumulative
    .sort_values(
        ['player_id', 'performance_window_start']
    )
    .reset_index(drop=True)
)

temporal_dataset = (
    temporal_dataset
    .sort_values(
        ['player_id', 'snapshot_date']
    )
    .reset_index(drop=True)
)

# Attach cumulative values
for col in cum_columns:

    temporal_dataset[
        f'{col}_at_snapshot'
    ] = snapshot_cumulative[col].fillna(0).values

    temporal_dataset[
        f'{col}_at_window_start'
    ] = window_cumulative[
        f'{col}_window_start'
    ].fillna(0).values


# Calculate performance inside the 6-month window
temporal_dataset['games_last_6m'] = (
    temporal_dataset['cum_games_at_snapshot']
    - temporal_dataset['cum_games_at_window_start']
)

temporal_dataset['goals_last_6m'] = (
    temporal_dataset['cum_goals_at_snapshot']
    - temporal_dataset['cum_goals_at_window_start']
)

temporal_dataset['assists_last_6m'] = (
    temporal_dataset['cum_assists_at_snapshot']
    - temporal_dataset['cum_assists_at_window_start']
)

temporal_dataset['minutes_last_6m'] = (
    temporal_dataset['cum_minutes_played_at_snapshot']
    - temporal_dataset['cum_minutes_played_at_window_start']
)

temporal_dataset['yellow_cards_last_6m'] = (
    temporal_dataset['cum_yellow_cards_at_snapshot']
    - temporal_dataset['cum_yellow_cards_at_window_start']
)

temporal_dataset['red_cards_last_6m'] = (
    temporal_dataset['cum_red_cards_at_snapshot']
    - temporal_dataset['cum_red_cards_at_window_start']
)

performance_features = [
    'games_last_6m',
    'goals_last_6m',
    'assists_last_6m',
    'minutes_last_6m',
    'yellow_cards_last_6m',
    'red_cards_last_6m'
]

print("=" * 70)
print("LAST SIX-MONTH PERFORMANCE FEATURES")
print("=" * 70)

display(
    temporal_dataset[
        [
            'player_id',
            'performance_window_start',
            'snapshot_date'
        ]
        + performance_features
    ].head(15)
)

LAST SIX-MONTH PERFORMANCE FEATURES


,player_id,performance_window_start,snapshot_date,games_last_6m,goals_last_6m,assists_last_6m,minutes_last_6m,yellow_cards_last_6m,red_cards_last_6m
0,10,2012-07-14,2013-01-14,22.00,11.00,2.00,"1,660.00",6.00,0.00
1,10,2012-12-19,2013-06-19,17.00,5.00,2.00,"1,192.00",2.00,0.00
2,10,2013-07-07,2014-01-07,14.00,5.00,2.00,"1,040.00",1.00,0.00
3,10,2014-01-07,2014-07-07,15.00,3.00,3.00,"1,180.00",1.00,0.00
4,10,2014-07-07,2015-01-07,15.00,4.00,3.00,496.00,2.00,0.00
5,10,2015-01-01,2015-07-01,25.00,12.00,6.00,"1,793.00",4.00,0.00
6,26,2012-07-13,2013-01-13,27.00,0.00,0.00,"2,430.00",1.00,0.00
7,26,2014-01-10,2014-07-10,22.00,0.00,0.00,"2,010.00",0.00,0.00
8,26,2014-08-04,2015-02-04,19.00,0.00,0.00,"1,710.00",0.00,0.00
9,26,2015-01-01,2015-07-01,15.00,0.00,0.00,"1,350.00",0.00,0.00


In [19]:
# ============================================================
# Validate Historical Performance Features
# ============================================================

print("=" * 70)
print("HISTORICAL PERFORMANCE VALIDATION")
print("=" * 70)

print(
    f"Dataset rows : "
    f"{len(temporal_dataset):,}"
)

print("\nMissing values:")

display(
    temporal_dataset[
        performance_features
    ].isna().sum()
)

negative_counts = (
    temporal_dataset[
        performance_features
    ]
    .lt(0)
    .sum()
)

print("\nNegative values:")

display(
    negative_counts
)

print("\nPerformance statistics:")

display(
    temporal_dataset[
        performance_features
    ].describe()
)

HISTORICAL PERFORMANCE VALIDATION
Dataset rows : 213,905

Missing values:


games_last_6m           0
goals_last_6m           0
assists_last_6m         0
minutes_last_6m         0
yellow_cards_last_6m    0
red_cards_last_6m       0
dtype: int64


Negative values:


games_last_6m           0
goals_last_6m           0
assists_last_6m         0
minutes_last_6m         0
yellow_cards_last_6m    0
red_cards_last_6m       0
dtype: int64


Performance statistics:


,games_last_6m,goals_last_6m,assists_last_6m,minutes_last_6m,yellow_cards_last_6m,red_cards_last_6m
count,"213,905.00","213,905.00","213,905.00","213,905.00","213,905.00","213,905.00"
mean,7.60,0.77,0.60,525.95,1.11,0.03
std,8.21,1.94,1.35,634.59,1.74,0.17
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,0.00,0.00,0.00
50%,5.00,0.00,0.00,220.00,0.00,0.00
75%,14.00,1.00,1.00,965.00,2.00,0.00
max,42.00,40.00,21.00,"3,422.00",16.00,3.00


In [20]:
# ============================================================
# Manual Temporal Leakage Check
# ============================================================

# Select a snapshot with recorded performance
sample_row = (
    temporal_dataset[
        temporal_dataset['games_last_6m'] > 0
    ]
    .iloc[0]
)

sample_player = sample_row['player_id']
sample_start = sample_row['performance_window_start']
sample_snapshot = sample_row['snapshot_date']

# Manually retrieve ONLY appearances inside the historical window
manual_appearances = df_appearances[
    (df_appearances['player_id'] == sample_player)
    &
    (df_appearances['date'] > sample_start)
    &
    (df_appearances['date'] <= sample_snapshot)
].copy()

manual_results = {
    'games_last_6m': len(manual_appearances),
    'goals_last_6m': manual_appearances['goals'].sum(),
    'assists_last_6m': manual_appearances['assists'].sum(),
    'minutes_last_6m': manual_appearances['minutes_played'].sum(),
    'yellow_cards_last_6m': manual_appearances['yellow_cards'].sum(),
    'red_cards_last_6m': manual_appearances['red_cards'].sum()
}

print("=" * 70)
print("MANUAL TEMPORAL LEAKAGE CHECK")
print("=" * 70)

print(f"Player ID    : {sample_player}")
print(f"Window start : {sample_start.date()}")
print(f"Snapshot     : {sample_snapshot.date()}")

print("\nPipeline vs Manual Calculation")
print("-" * 70)

all_match = True

for feature in performance_features:

    pipeline_value = sample_row[feature]
    manual_value = manual_results[feature]

    match = np.isclose(
        pipeline_value,
        manual_value
    )

    all_match = all_match and match

    print(
        f"{feature:<25} "
        f"Pipeline = {pipeline_value:>8.0f} | "
        f"Manual = {manual_value:>8.0f} | "
        f"Match = {match}"
    )

print("\n" + "=" * 70)

print(
    "ALL FEATURES MATCH:",
    all_match
)

print("=" * 70)

MANUAL TEMPORAL LEAKAGE CHECK
Player ID    : 10
Window start : 2012-07-14
Snapshot     : 2013-01-14

Pipeline vs Manual Calculation
----------------------------------------------------------------------
games_last_6m             Pipeline =       22 | Manual =       22 | Match = True
goals_last_6m             Pipeline =       11 | Manual =       11 | Match = True
assists_last_6m           Pipeline =        2 | Manual =        2 | Match = True
minutes_last_6m           Pipeline =     1660 | Manual =     1660 | Match = True
yellow_cards_last_6m      Pipeline =        6 | Manual =        6 | Match = True
red_cards_last_6m         Pipeline =        0 | Manual =        0 | Match = True

ALL FEATURES MATCH: True


In [21]:
# ============================================================
# Explicit Future Leakage Check
# ============================================================

future_appearance_count = (
    manual_appearances['date']
    > sample_snapshot
).sum()

before_or_at_snapshot = (
    manual_appearances['date']
    <= sample_snapshot
).all()

inside_window = (
    (
        manual_appearances['date']
        > sample_start
    )
    &
    (
        manual_appearances['date']
        <= sample_snapshot
    )
).all()

print("=" * 70)
print("FUTURE LEAKAGE CHECK")
print("=" * 70)

print(
    f"Future appearances included : "
    f"{future_appearance_count}"
)

print(
    f"All appearances <= snapshot : "
    f"{before_or_at_snapshot}"
)

print(
    f"All appearances inside window: "
    f"{inside_window}"
)

print("\nAppearance date range used:")

print(
    manual_appearances['date'].min(),
    "→",
    manual_appearances['date'].max()
)

FUTURE LEAKAGE CHECK
Future appearances included : 0
All appearances <= snapshot : True
All appearances inside window: True

Appearance date range used:
2012-08-23 00:00:00 → 2013-01-13 00:00:00


In [ ]:
# ============================================================
# Add Temporal-Safe Static Player Features
# ============================================================

safe_static_columns = [
    'player_id',
    'position',
    'sub_position',
    'foot',
    'height_in_cm',
    'country_of_citizenship'
]

player_static_features = (
    df_players[
        safe_static_columns
    ]
    .drop_duplicates(
        subset='player_id'
    )
    .copy()
)

temporal_dataset = temporal_dataset.merge(
    player_static_features,
    on='player_id',
    how='left'
)

print("=" * 70)
print("TEMPORAL-SAFE STATIC FEATURES")
print("=" * 70)

print(
    f"Dataset rows : "
    f"{len(temporal_dataset):,}"
)

print("\nMissing values:")

display(
    temporal_dataset[
        [
            'age_at_snapshot',
            'position',
            'sub_position',
            'foot',
            'height_in_cm',
            'country_of_citizenship'
        ]
    ]
    .isna()
    .sum()
)

display(
    temporal_dataset[
        [
            'player_id',
            'snapshot_date',
            'age_at_snapshot',
            'position',
            'sub_position',
            'foot',
            'height_in_cm',
            'country_of_citizenship'
        ]
    ].head(10)
)

TEMPORAL-SAFE STATIC FEATURES
Dataset rows : 213,905

Missing values:


age_at_snapshot            203
position                     0
sub_position               214
foot                      3030
height_in_cm              2052
country_of_citizenship    1531
dtype: int64

,player_id,snapshot_date,age_at_snapshot,position,sub_position,foot,height_in_cm,country_of_citizenship
0,10,2013-01-14,34.60,Attack,Centre-Forward,right,184.00,Germany
1,10,2013-06-19,35.03,Attack,Centre-Forward,right,184.00,Germany
2,10,2014-01-07,35.58,Attack,Centre-Forward,right,184.00,Germany
3,10,2014-07-07,36.08,Attack,Centre-Forward,right,184.00,Germany
4,10,2015-01-07,36.58,Attack,Centre-Forward,right,184.00,Germany
5,10,2015-07-01,37.06,Attack,Centre-Forward,right,184.00,Germany
6,26,2013-01-13,32.44,Goalkeeper,Goalkeeper,left,190.00,Germany
7,26,2014-07-10,33.92,Goalkeeper,Goalkeeper,left,190.00,Germany
8,26,2015-02-04,34.50,Goalkeeper,Goalkeeper,left,190.00,Germany
9,26,2015-07-01,34.90,Goalkeeper,Goalkeeper,left,190.00,Germany


In [23]:
# ============================================================
# Create Recent Performance Rate Features
# ============================================================

MIN_MINUTES_FOR_RATE = 90

temporal_dataset['goals_per_90_last_6m'] = np.where(
    temporal_dataset['minutes_last_6m']
        >= MIN_MINUTES_FOR_RATE,

    temporal_dataset['goals_last_6m']
    / temporal_dataset['minutes_last_6m']
    * 90,

    np.nan
)

temporal_dataset['assists_per_90_last_6m'] = np.where(
    temporal_dataset['minutes_last_6m']
        >= MIN_MINUTES_FOR_RATE,

    temporal_dataset['assists_last_6m']
    / temporal_dataset['minutes_last_6m']
    * 90,

    np.nan
)

temporal_dataset[
    'goal_contribution_per_90_last_6m'
] = np.where(
    temporal_dataset['minutes_last_6m']
        >= MIN_MINUTES_FOR_RATE,

    (
        temporal_dataset['goals_last_6m']
        + temporal_dataset['assists_last_6m']
    )
    / temporal_dataset['minutes_last_6m']
    * 90,

    np.nan
)

temporal_dataset['minutes_per_game_last_6m'] = np.where(
    temporal_dataset['games_last_6m'] > 0,

    temporal_dataset['minutes_last_6m']
    / temporal_dataset['games_last_6m'],

    np.nan
)

rate_features = [
    'goals_per_90_last_6m',
    'assists_per_90_last_6m',
    'goal_contribution_per_90_last_6m',
    'minutes_per_game_last_6m'
]

print("=" * 70)
print("RECENT PERFORMANCE RATE FEATURES")
print("=" * 70)

print(
    f"Minimum minutes for per-90 rates : "
    f"{MIN_MINUTES_FOR_RATE}"
)

print("\nMissing values:")

display(
    temporal_dataset[
        rate_features
    ].isna().sum()
)

print("\nRate statistics:")

display(
    temporal_dataset[
        rate_features
    ].describe()
)

RECENT PERFORMANCE RATE FEATURES
Minimum minutes for per-90 rates : 90

Missing values:


goals_per_90_last_6m                92222
assists_per_90_last_6m              92222
goal_contribution_per_90_last_6m    92222
minutes_per_game_last_6m            79865
dtype: int64


Rate statistics:


,goals_per_90_last_6m,assists_per_90_last_6m,goal_contribution_per_90_last_6m,minutes_per_game_last_6m
count,"121,683.00","121,683.00","121,683.00","134,040.00"
mean,0.13,0.10,0.23,63.94
std,0.21,0.16,0.29,24.18
min,0.00,0.00,0.00,1.00
25%,0.00,0.00,0.00,47.69
50%,0.00,0.00,0.12,70.45
75%,0.19,0.15,0.35,84.70
max,2.90,2.57,4.04,120.00


In [24]:
# ============================================================
# Create Future Value Change Variables
# Analysis Only - NOT Model Features
# ============================================================

temporal_dataset['future_value_change_eur'] = (
    temporal_dataset['future_market_value']
    - temporal_dataset['current_market_value']
)

temporal_dataset['future_value_change_pct'] = (
    temporal_dataset['future_value_change_eur']
    / temporal_dataset['current_market_value']
    * 100
)

print("=" * 70)
print("FUTURE MARKET VALUE CHANGE")
print("=" * 70)

print("These variables are for analysis only.")
print("They MUST NOT be used as predictor features.\n")

display(
    temporal_dataset[
        [
            'current_market_value',
            'future_market_value',
            'future_value_change_eur',
            'future_value_change_pct'
        ]
    ].describe()
)

FUTURE MARKET VALUE CHANGE
These variables are for analysis only.
They MUST NOT be used as predictor features.



,current_market_value,future_market_value,future_value_change_eur,future_value_change_pct
count,"213,905.00","213,905.00","213,905.00","213,905.00"
mean,"3,744,025.57","3,905,672.07","161,646.50",10.26
std,"9,104,194.97","9,595,893.33","2,702,548.56",104.22
min,"10,000.00","10,000.00","-60,000,000.00",-95.38
25%,"350,000.00","350,000.00","-150,000.00",-20.00
50%,"800,000.00","800,000.00",0.00,0.00
75%,"3,000,000.00","3,000,000.00","50,000.00",12.50
max,"200,000,000.00","200,000,000.00","110,000,000.00","14,900.00"


In [25]:
# ============================================================
# Final Temporal Feature Inventory
# ============================================================

numeric_features = [
    'current_market_value',
    'age_at_snapshot',

    # Recent 6-month performance
    'games_last_6m',
    'goals_last_6m',
    'assists_last_6m',
    'minutes_last_6m',
    'yellow_cards_last_6m',
    'red_cards_last_6m',

    # Recent performance rates
    'goals_per_90_last_6m',
    'assists_per_90_last_6m',
    'goal_contribution_per_90_last_6m',
    'minutes_per_game_last_6m',

    # Static numeric attribute
    'height_in_cm'
]

categorical_features = [
    'position',
    'sub_position',
    'foot',
    'country_of_citizenship'
]

target = 'future_market_value'

analysis_only = [
    'future_value_change_eur',
    'future_value_change_pct'
]

print("=" * 70)
print("FINAL TEMPORAL FEATURE INVENTORY")
print("=" * 70)

print(f"\nNumeric features ({len(numeric_features)}):")
for feature in numeric_features:
    print(f"  - {feature}")

print(f"\nCategorical features ({len(categorical_features)}):")
for feature in categorical_features:
    print(f"  - {feature}")

print(f"\nTotal candidate predictors: "
      f"{len(numeric_features) + len(categorical_features)}")

print(f"\nTarget:")
print(f"  - {target}")

print("\nAnalysis only - NOT predictors:")
for feature in analysis_only:
    print(f"  - {feature}")

FINAL TEMPORAL FEATURE INVENTORY

Numeric features (13):
  - current_market_value
  - age_at_snapshot
  - games_last_6m
  - goals_last_6m
  - assists_last_6m
  - minutes_last_6m
  - yellow_cards_last_6m
  - red_cards_last_6m
  - goals_per_90_last_6m
  - assists_per_90_last_6m
  - goal_contribution_per_90_last_6m
  - minutes_per_game_last_6m
  - height_in_cm

Categorical features (4):
  - position
  - sub_position
  - foot
  - country_of_citizenship

Total candidate predictors: 17

Target:
  - future_market_value

Analysis only - NOT predictors:
  - future_value_change_eur
  - future_value_change_pct


In [26]:
# ============================================================
# Create Final Temporal Machine Learning Dataset
# ============================================================

metadata_columns = [
    'player_id',
    'snapshot_date',
    'target_date',
    'future_date',
    'target_distance_days',
    'performance_window_start'
]

final_columns = (
    metadata_columns
    + numeric_features
    + categorical_features
    + [target]
    + analysis_only
)

df_temporal_final = temporal_dataset[
    final_columns
].copy()

# Sort chronologically
df_temporal_final = (
    df_temporal_final
    .sort_values(
        ['snapshot_date', 'player_id']
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("FINAL TEMPORAL ML DATASET")
print("=" * 70)

print(
    f"Rows    : "
    f"{len(df_temporal_final):,}"
)

print(
    f"Columns : "
    f"{df_temporal_final.shape[1]}"
)

print(
    f"Players : "
    f"{df_temporal_final['player_id'].nunique():,}"
)

print("\nSnapshot range:")

print(
    df_temporal_final['snapshot_date'].min(),
    "→",
    df_temporal_final['snapshot_date'].max()
)

print("\nTarget date range:")

print(
    df_temporal_final['future_date'].min(),
    "→",
    df_temporal_final['future_date'].max()
)

display(
    df_temporal_final.head()
)

FINAL TEMPORAL ML DATASET
Rows    : 213,905
Columns : 26
Players : 24,275

Snapshot range:
2012-07-06 00:00:00 → 2025-12-22 00:00:00

Target date range:
2012-12-29 00:00:00 → 2026-05-18 00:00:00


,player_id,snapshot_date,target_date,future_date,target_distance_days,performance_window_start,current_market_value,age_at_snapshot,games_last_6m,goals_last_6m,assists_last_6m,minutes_last_6m,yellow_cards_last_6m,red_cards_last_6m,goals_per_90_last_6m,assists_per_90_last_6m,goal_contribution_per_90_last_6m,minutes_per_game_last_6m,height_in_cm,position,sub_position,foot,country_of_citizenship,future_market_value,future_value_change_eur,future_value_change_pct
0,4520,2012-07-06,2013-01-06,2013-01-12,6,2012-01-06,2500000,32.63,1.00,0.00,0.00,90.00,0.00,0.00,0.00,0.00,0.00,90.00,181.00,Defender,Centre-Back,right,Netherlands,2000000,-500000,-20.00
1,30003,2012-07-06,2013-01-06,2013-01-12,6,2012-01-06,5000000,25.88,1.00,0.00,0.00,90.00,0.00,0.00,0.00,0.00,0.00,90.00,176.00,Midfield,Defensive Midfield,right,NaN,4500000,-500000,-10.00
2,47740,2012-07-06,2013-01-06,2013-01-17,11,2012-01-06,400000,20.40,1.00,0.00,0.00,90.00,0.00,0.00,0.00,0.00,0.00,90.00,190.00,Defender,Centre-Back,both,Germany,600000,200000,50.00
3,60312,2012-07-06,2013-01-06,2013-01-17,11,2012-01-06,600000,23.76,1.00,1.00,1.00,45.00,0.00,0.00,NaN,NaN,NaN,45.00,175.00,Attack,Left Winger,right,Aruba,1250000,650000,108.33
4,80073,2012-07-06,2013-01-06,2013-01-09,3,2012-01-06,1500000,25.43,1.00,2.00,1.00,90.00,0.00,0.00,2.00,1.00,3.00,90.00,194.00,Attack,Centre-Forward,right,Netherlands,2000000,500000,33.33


In [27]:
# ============================================================
# Final Temporal Dataset Quality Check
# ============================================================

print("=" * 70)
print("FINAL TEMPORAL DATA QUALITY CHECK")
print("=" * 70)

print(
    f"Duplicate player-snapshot rows : "
    f"{df_temporal_final.duplicated(
        subset=['player_id', 'snapshot_date']
    ).sum():,}"
)

print(
    f"Missing target values          : "
    f"{df_temporal_final[target].isna().sum():,}"
)

print(
    f"Non-positive target values     : "
    f"{(df_temporal_final[target] <= 0).sum():,}"
)

print(
    f"Future dates <= snapshot       : "
    f"{(
        df_temporal_final['future_date']
        <= df_temporal_final['snapshot_date']
    ).sum():,}"
)

print("\nMissing values in candidate predictors:")

missing_summary = (
    df_temporal_final[
        numeric_features
        + categorical_features
    ]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    pd.DataFrame({
        'Missing_Count': missing_summary,
        'Missing_Percent': (
            missing_summary
            / len(df_temporal_final)
            * 100
        )
    })
)

FINAL TEMPORAL DATA QUALITY CHECK
Duplicate player-snapshot rows : 0
Missing target values          : 0
Non-positive target values     : 0
Future dates <= snapshot       : 0

Missing values in candidate predictors:


,Missing_Count,Missing_Percent
goals_per_90_last_6m,92222,43.11
goal_contribution_per_90_last_6m,92222,43.11
assists_per_90_last_6m,92222,43.11
minutes_per_game_last_6m,79865,37.34
foot,3030,1.42
height_in_cm,2052,0.96
country_of_citizenship,1531,0.72
sub_position,214,0.10
age_at_snapshot,203,0.09
current_market_value,0,0.00


In [28]:
# ============================================================
# Save Final Temporal Dataset
# ============================================================

OUTPUT_PATH = (
    '../../../data/processed/'
    'temporal_market_value_dataset.csv'
)

df_temporal_final.to_csv(
    OUTPUT_PATH,
    index=False
)

print("=" * 70)
print("TEMPORAL DATASET SAVED")
print("=" * 70)

print(f"Path    : {OUTPUT_PATH}")
print(f"Rows    : {len(df_temporal_final):,}")
print(f"Columns : {df_temporal_final.shape[1]}")

TEMPORAL DATASET SAVED
Path    : ../../../data/processed/temporal_market_value_dataset.csv
Rows    : 213,905
Columns : 26
